In [1]:
from pyspark.sql import SparkSession

In [3]:
df = spark.read.csv(
    "hdfs://localhost:9000/hotel_project/input/*",
    header=True,
    inferSchema=True
)

In [4]:
df.show(5)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+----+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type| adr|required_car_parking_spaces|total_

In [5]:
df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

In [6]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 119390
Columns: 32


In [7]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required_car_parking_spaces|total_of_special_reque

In [8]:
df = df.fillna({"children": 0})

In [9]:
df = df.dropna()

In [10]:
print(df.count())

119390


In [14]:
df = df.dropDuplicates()

In [15]:
print("Rows after removing duplicates:", df.count())

Rows after removing duplicates: 87396


In [16]:
df = df.withColumn(
    "total_nights",
    col("stays_in_weekend_nights") + col("stays_in_week_nights")
)

In [17]:
df = df.withColumn(
    "total_guests",
    col("adults") + col("children") + col("babies")
)

In [18]:
df = df.withColumn(
    "total_stay_cost",
    col("adr") * col("total_nights")
)

In [19]:
df.select(
    "hotel",
    "adr",
    "total_nights",
    "total_guests",
    "total_stay_cost"
).show(10)


+------------+------+------------+------------+------------------+
|       hotel|   adr|total_nights|total_guests|   total_stay_cost|
+------------+------+------------+------------+------------------+
|Resort Hotel|134.73|          14|         2.0|1886.2199999999998|
|Resort Hotel|  79.5|           1|         2.0|              79.5|
|Resort Hotel|  73.8|           2|         2.0|             147.6|
|Resort Hotel|121.01|           7|         2.0|            847.07|
|Resort Hotel| 133.2|           4|         2.0|             532.8|
|Resort Hotel| 223.0|           2|         2.0|             446.0|
|Resort Hotel|151.95|           6|         2.0| 911.6999999999999|
|Resort Hotel| 137.7|           1|         2.0|             137.7|
|Resort Hotel| 87.75|          10|         2.0|             877.5|
|Resort Hotel| 69.83|           7|         2.0|            488.81|
+------------+------+------------+------------+------------------+
only showing top 10 rows



In [20]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 87396
Columns: 35


In [28]:
df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://localhost:9000/hotel_project/cleaned_data_final")

Duplicates: 0


+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+----+---------------------------+-------------------------+------------------+-----------------------+------------+------------+---------------+
|hotel     |is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+------------+------------+---------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required

Duplicates: 0
